EYT-Net uses two detection scales. Good anchor boxes should match the actual shapes in our data instead of defaults

We follow the usual YOLO practice as we researched:
- cluster on width/height only
- use 1 - IoU distance
- fit on train only, at 640×640 letterbox scale
- drop near-zero boxes

In [1]:
import sys
from pathlib import Path
import numpy as np
from PIL import Image
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
DATA_DIR = PROJECT_ROOT / "data"
IMG_SIZE = 640

In [3]:
box_wh = []

for label_path in (DATA_DIR / "train" / "labels").glob("*.txt"): #train only
    text = label_path.read_text().strip()
    if not text:
        continue
    img_path = DATA_DIR / "train" / "images" / f"{label_path.stem}.jpg"
    orig_w, orig_h = Image.open(img_path).size
    scale = min(IMG_SIZE / orig_w, IMG_SIZE / orig_h)

    for line in text.splitlines():
        _, _, _, w, h = map(float, line.split())
        # convert normalized size to pixels at letterboxed resolution
        px_w = w * orig_w * scale
        px_h = h * orig_h * scale
        box_wh.append((px_w, px_h))

In [6]:
box_wh = np.array(box_wh)

In [9]:
len(box_wh), box_wh[:, 0].min(), box_wh[:, 0].max(), box_wh[:, 1].min(), box_wh[:, 1].max()

(4812,
 np.float64(3.5000000000000004),
 np.float64(639.5),
 np.float64(5.0),
 np.float64(359.5))

4812 train boxes at the 640 letterbox scale. Widths run 3.5–640 px, heights 5–360 px. so we have tiny objects and nearly full frame ones in the same set. That spread is exactly why dataset based anchors matter more than COCO defaults

Normal k-means uses Euclidean distance but that pulls the centers toward big boxes and ignores the small ones. For anchors we care about shape overlap so we measure distance as 1 - IoU as YOLO best practices show

All boxes are treated as if they share the same origin we only compare width and height

In [10]:
def box_iou(box, clusters):
    inter_w = np.minimum(clusters[:, 0], box[0])
    inter_h = np.minimum(clusters[:, 1], box[1])
    inter = inter_w * inter_h
    union = box[0] * box[1] + clusters[:, 0] * clusters[:, 1] - inter
    return inter / union

For each ground-truth box we take its best match to any cluster then average those scores. Higher = anchors cover the dataset better.

In [11]:
def avg_iou(boxes, clusters):
    return np.mean([np.max(box_iou(box, clusters)) for box in boxes])

Now we will implement k-means with IoU distance

In [ ]:
def kmeans_iou(boxes, k, max_iter=100):
    rng = np.random.default_rng()
    centers = boxes[rng.choice(len(boxes), size=k, replace=False)].copy()

    for _ in range(max_iter):
        dist = 1 - np.array([1 - box_iou(box, centers) for box in boxes])
        labels = np.argmin(dist, axis=1)

        new_centers = centers.copy()
        for i in range(k):
            members = boxes[labels == i]
            if len(members):
                new_centers[i] = np.median(members, axis=0) # for outliers median is better than mean. We use because box sizes are skewed